# Benchmark analytics for deterministic, full-HTML LLM, and hybrid extraction

This notebook is the reproducible analysis layer for the three benchmark arms:

- `original_fast` — deterministic extraction;
- `llm_full_html` — full-HTML LLM baseline;
- `hybrid_original` — coverage-aware deterministic-first hybrid.

It discovers the review workbooks under the configured result directories, validates comparability, derives metrics from the underlying `AllResults` and `Pages` tables, and exports publication-ready figures and CSV tables.

**Scientific guardrails.** The notebook does not silently merge incompatible ground-truth sets. Paired analyses are emitted only for pages shared by the compared variants. `tailored` / `untailored` is taken from the `Domens` sheet and used only for post-hoc analysis, never as a routing signal. Cost values below are externally reported billing values and are kept separate from benchmark-derived metrics.

In [18]:
%pip install "matplotlib==3.11.2"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from __future__ import annotations

import json
import math
import os
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

# Exact paths supplied for the benchmark repository. Environment variables make
# the notebook portable for CI / another machine without editing cells.
RESULT_ROOTS = {
    "original_fast": Path(os.environ.get(
        "BENCH_DETERMINED_DIR",
        r"D:\Pet_Codes\science\article-price-tracker\baseline_results\determined",
    )),
    "llm_full_html": Path(os.environ.get(
        "BENCH_LLM_DIR",
        r"D:\Pet_Codes\science\article-price-tracker\baseline_results\llm\qwen38-omni-flash-v2",
    )),
    "hybrid_original": Path(os.environ.get(
        "BENCH_HYBRID_DIR",
        r"D:\Pet_Codes\science\article-price-tracker\baseline_results\hybrid\qwen38-hybrid-coverage-v2",
    )),
}

REPO_ROOT = Path(os.environ.get(
    "BENCH_REPO_ROOT",
    r"D:\Pet_Codes\science\article-price-tracker",
))
OUTPUT_ROOT = Path(os.environ.get("BENCH_ANALYTICS_OUTPUT", str(REPO_ROOT / "analytics_output")))
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

DISPLAY_NAMES = {
    "original_fast": "Deterministic",
    "llm_full_html": "Full-HTML LLM",
    "hybrid_original": "Hybrid",
}
DATASET_ORDER = ["synthetic", "robustness", "industrial"]
VARIANT_ORDER = ["original_fast", "hybrid_original", "llm_full_html"]

# Exact billing figures reported by the experimenter for the article runs.
# Currency conversion uses the contemporaneous experimental rate supplied by
# the experimenter: 83.3 RUB per USD. These values are not model estimates.
RUB_PER_USD = 83.3
COST_OVERRIDES = pd.DataFrame([
    {"variant":"original_fast", "total_tokens":0, "cost_rub":0.0, "cost_source":"deterministic; no LLM inference"},
    {"variant":"llm_full_html", "total_tokens":57_923_054, "cost_rub":995.50, "cost_source":"exact reported provider billing for the experiment"},
    {"variant":"hybrid_original", "total_tokens":3_235_450, "cost_rub":60.56, "cost_source":"exact reported provider billing for the experiment"},
])

BOOTSTRAP_SEED = 20261005
BOOTSTRAP_N = 3000

print("Output:", OUTPUT_ROOT)
for k, p in RESULT_ROOTS.items():
    print(f"{k:16s} -> {p}  exists={p.exists()}")


Output: D:\Pet_Codes\science\article-price-tracker\analytics_output
original_fast    -> D:\Pet_Codes\science\article-price-tracker\baseline_results\determined  exists=True
llm_full_html    -> D:\Pet_Codes\science\article-price-tracker\baseline_results\llm\qwen38-omni-flash-v2  exists=True
hybrid_original  -> D:\Pet_Codes\science\article-price-tracker\baseline_results\hybrid\qwen38-hybrid-coverage-v2  exists=True


## 1. Workbook discovery and schema loading

The benchmark directories can contain checkpoints, caches, and multiple Excel exports. The discovery function prefers a `baseline_review_all*.xlsx` workbook, then `baseline_comparison*.xlsx`, and otherwise chooses the workbook with the richest benchmark schema. Each arm is filtered to its expected variant, so a combined comparison workbook cannot accidentally duplicate rows.

In [20]:
REQUIRED_SHEETS = {"Metrics", "AllResults", "Pages"}


def workbook_sheet_names(path: Path) -> set[str]:
    try:
        return set(pd.ExcelFile(path).sheet_names)
    except Exception:
        return set()


def workbook_score(path: Path) -> tuple[int, int, float]:
    sheets = workbook_sheet_names(path)
    if not REQUIRED_SHEETS.issubset(sheets):
        return (-10_000, 0, 0.0)
    name = path.name.lower()
    score = 0
    if "baseline_review_all" in name:
        score += 1000
    if "baseline_comparison" in name:
        score += 800
    if "review" in name:
        score += 200
    score += len(sheets) * 10
    try:
        metric_rows = len(pd.read_excel(path, sheet_name="Metrics", usecols=["dataset", "variant"]))
    except Exception:
        metric_rows = 0
    try:
        mtime = path.stat().st_mtime
    except OSError:
        mtime = 0.0
    return (score, metric_rows, mtime)


def discover_workbook(root: Path, expected_variant: str) -> Path:
    if not root.exists():
        raise FileNotFoundError(f"Benchmark directory does not exist: {root}")
    candidates = [p for p in root.rglob("*.xlsx") if not p.name.startswith("~$")]
    valid = [p for p in candidates if REQUIRED_SHEETS.issubset(workbook_sheet_names(p))]
    if not valid:
        raise FileNotFoundError(
            f"No .xlsx workbook with sheets {sorted(REQUIRED_SHEETS)} found under {root}"
        )
    # Prefer workbooks that actually contain the expected variant.
    with_variant = []
    for p in valid:
        try:
            m = pd.read_excel(p, sheet_name="Metrics", usecols=["variant"])
            if expected_variant in set(m["variant"].astype(str)):
                with_variant.append(p)
        except Exception:
            pass
    pool = with_variant or valid
    return max(pool, key=workbook_score)


def read_optional_sheet(path: Path, sheet: str) -> pd.DataFrame:
    if sheet not in workbook_sheet_names(path):
        return pd.DataFrame()
    return pd.read_excel(path, sheet_name=sheet)


def load_arm(root: Path, expected_variant: str) -> dict:
    wb = discover_workbook(root, expected_variant)
    arm = {"workbook": wb}
    for sheet in ["Metrics", "AllResults", "Pages", "Comparison", "UnmatchedPredictions", "Domens", "Provenance"]:
        df = read_optional_sheet(wb, sheet)
        if not df.empty and "variant" in df.columns and sheet not in {"Provenance", "Domens"}:
            df = df[df["variant"].astype(str) == expected_variant].copy()
        arm[sheet] = df
    if arm["Metrics"].empty:
        raise ValueError(f"Expected variant {expected_variant!r} not found in Metrics of {wb}")
    return arm


arms = {v: load_arm(RESULT_ROOTS[v], v) for v in VARIANT_ORDER}
for variant, arm in arms.items():
    print(f"\n{variant}: {arm['workbook']}")
    print(" metrics rows:", len(arm["Metrics"]), "all-results rows:", len(arm["AllResults"]), "pages:", len(arm["Pages"]))
    print(" datasets:", sorted(arm["Metrics"]["dataset"].dropna().astype(str).unique().tolist()))


original_fast: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\baseline_review_all.xlsx
 metrics rows: 3 all-results rows: 11210 pages: 460
 datasets: ['industrial', 'robustness', 'synthetic']

hybrid_original: D:\Pet_Codes\science\article-price-tracker\baseline_results\hybrid\qwen38-hybrid-coverage-v2\baseline_review_all.xlsx
 metrics rows: 3 all-results rows: 11212 pages: 460
 datasets: ['industrial', 'robustness', 'synthetic']

llm_full_html: D:\Pet_Codes\science\article-price-tracker\baseline_results\llm\qwen38-omni-flash-v2\baseline_review_all.xlsx
 metrics rows: 3 all-results rows: 11010 pages: 460
 datasets: ['industrial', 'robustness', 'synthetic']


## 2. Harmonization, domain labels, and comparability checks

`Domens` is discovered from any selected workbook that contains it. The mapping is then applied uniformly to all three arms by `source`, avoiding dependence on manually copied Excel formulas.

In [21]:
# Locate the domain-tailoring mapping in any selected workbook.
domains = pd.DataFrame()
for arm in arms.values():
    d = arm.get("Domens", pd.DataFrame())
    if not d.empty and {"source", "development_group"}.issubset(d.columns):
        domains = d[["source", "development_group"]].dropna().drop_duplicates().copy()
        break

if domains.empty:
    warnings.warn("No Domens sheet found. tailored/untailored analyses will be skipped.")
    domain_map = {}
else:
    domains["source"] = domains["source"].astype(str).str.strip()
    domains["development_group"] = domains["development_group"].astype(str).str.strip()
    domain_map = dict(zip(domains["source"], domains["development_group"]))
    print(domains["development_group"].value_counts(dropna=False))


def normalize_bool_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(int)
    return s.map(lambda x: 1 if x is True or str(x).strip().lower() in {"1","true","yes"} else 0)


def attach_development_group(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "source" in df.columns and domain_map:
        mapped = df["source"].astype(str).str.strip().map(domain_map)
        if "development_group" in df.columns:
            existing = df["development_group"].astype("string").str.strip()
            df["development_group"] = mapped.fillna(existing)
        else:
            df["development_group"] = mapped
    return df

all_results_parts=[]
pages_parts=[]
metrics_parts=[]
for variant, arm in arms.items():
    a = attach_development_group(arm["AllResults"])
    p = attach_development_group(arm["Pages"])
    if not a.empty:
        a["variant"] = variant
        all_results_parts.append(a)
    if not p.empty:
        p["variant"] = variant
        pages_parts.append(p)
    m = arm["Metrics"].copy()
    m["variant"] = variant
    metrics_parts.append(m)

all_results = pd.concat(all_results_parts, ignore_index=True, sort=False)
pages = pd.concat(pages_parts, ignore_index=True, sort=False)
metrics = pd.concat(metrics_parts, ignore_index=True, sort=False)

for c in ["entity_TP","entity_FP","entity_FN","name_TP","name_FP","name_FN","price_TP","price_FP","price_FN"]:
    if c in all_results:
        all_results[c] = pd.to_numeric(all_results[c], errors="coerce").fillna(0)
    if c in pages:
        pages[c] = pd.to_numeric(pages[c], errors="coerce").fillna(0)

# Comparable-GT diagnostics: unique GT entities and page overlap.
rows=[]

for dataset in sorted(set(all_results["dataset"].dropna().astype(str))):
    for variant in VARIANT_ORDER:
        d = all_results[
            (all_results["dataset"].astype(str) == dataset) &
            (all_results["variant"] == variant)
        ]

        p = pages[
            (pages["dataset"].astype(str) == dataset) &
            (pages["variant"] == variant)
        ]

        gt_entities = (
            int(d["entity_TP"].sum() + d["entity_FN"].sum())
            if {"entity_TP", "entity_FN"}.issubset(d.columns)
            else np.nan
        )

        rows.append({
            "dataset": dataset,
            "variant": variant,
            "gt_entities": gt_entities,
            "pages": p["page_id"].dropna().astype(str).nunique()
                     if "page_id" in p.columns else np.nan,
            "sources": p["source"].dropna().astype(str).nunique()
                       if "source" in p.columns else np.nan,
        })

comparability = pd.DataFrame(rows)
comparability.to_csv(TABLE_DIR / "comparability_counts.csv", index=False)
display(comparability)

development_group
tailored      22
untailored     3
Name: count, dtype: int64


,dataset,variant,gt_entities,pages,sources
0,industrial,original_fast,6317,240,25
1,industrial,hybrid_original,6317,240,25
2,industrial,llm_full_html,6317,240,25
3,robustness,original_fast,4200,200,1
4,robustness,hybrid_original,4200,200,1
5,robustness,llm_full_html,4200,200,1
6,synthetic,original_fast,420,20,1
7,synthetic,hybrid_original,420,20,1
8,synthetic,llm_full_html,420,20,1


## 3. Core metric tables

All aggregate metrics below are recomputed from TP/FP/FN counts where possible. This avoids relying on formatted Excel formulas and provides a single calculation path for overall, development-group, domain, mutation-axis, and page-level analyses.

In [22]:
def prf(tp, fp, fn):
    tp, fp, fn = float(tp), float(fp), float(fn)
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f


def aggregate_prf(df: pd.DataFrame, group_cols: list[str], prefix="entity") -> pd.DataFrame:
    cols=[f"{prefix}_TP", f"{prefix}_FP", f"{prefix}_FN"]
    if not set(cols).issubset(df.columns):
        return pd.DataFrame()
    g=df.groupby(group_cols, dropna=False)[cols].sum().reset_index()
    if g.empty:
        return pd.DataFrame(columns=group_cols + cols + [f"{prefix}_P",f"{prefix}_R",f"{prefix}_F1"])
    values=g.apply(lambda r: prf(r[cols[0]],r[cols[1]],r[cols[2]]), axis=1, result_type="expand")
    values.columns=[f"{prefix}_P",f"{prefix}_R",f"{prefix}_F1"]
    return pd.concat([g,values],axis=1)

# Overall micro metrics from raw AllResults.
overall_entity = aggregate_prf(all_results, ["dataset","variant"], "entity")
overall_name = aggregate_prf(all_results, ["dataset","variant"], "name")
overall_price = aggregate_prf(all_results, ["dataset","variant"], "price")
overall = overall_entity.merge(overall_name, on=["dataset","variant"], how="outer").merge(overall_price,on=["dataset","variant"],how="outer")
overall["variant_label"] = overall["variant"].map(DISPLAY_NAMES)
overall.to_csv(TABLE_DIR / "overall_metrics.csv", index=False)
display(overall.sort_values(["dataset","variant"]))

# Development-group micro metrics for industrial pages.
industrial = all_results[all_results["dataset"].astype(str)=="industrial"].copy()
if "development_group" in industrial and industrial["development_group"].notna().any():
    dev_entity = aggregate_prf(industrial, ["variant","development_group"], "entity")
    dev_entity["variant_label"] = dev_entity["variant"].map(DISPLAY_NAMES)
    dev_entity.to_csv(TABLE_DIR / "development_split_metrics.csv", index=False)
    display(dev_entity)
else:
    dev_entity = pd.DataFrame()

# Per-domain micro metrics.
if "source" in industrial.columns:
    per_domain = aggregate_prf(industrial, ["variant","source","development_group"], "entity")
    per_domain["variant_label"] = per_domain["variant"].map(DISPLAY_NAMES)
    per_domain.to_csv(TABLE_DIR / "per_domain_metrics.csv", index=False)
else:
    per_domain = pd.DataFrame()

,dataset,variant,entity_TP,entity_FP,entity_FN,entity_P,entity_R,entity_F1,name_TP,name_FP,name_FN,name_P,name_R,name_F1,price_TP,price_FP,price_FN,price_P,price_R,price_F1,variant_label
0,industrial,hybrid_original,5851,275,466,0.955109,0.926231,0.940448,5821,305,496,0.950212,0.921482,0.935626,5511,615,646,0.899608,0.895079,0.897338,Hybrid
1,industrial,llm_full_html,5970,73,347,0.987920,0.945069,0.966019,5650,393,667,0.934966,0.894412,0.914239,5686,195,471,0.966842,0.923502,0.944675,Full-HTML LLM
2,industrial,original_fast,5799,273,518,0.955040,0.917999,0.936153,5789,283,528,0.953393,0.916416,0.934539,5460,612,697,0.899209,0.886796,0.892959,Deterministic
3,robustness,hybrid_original,3082,0,1118,1.000000,0.733810,0.846471,3082,0,1118,1.000000,0.733810,0.846471,3082,0,1118,1.000000,0.733810,0.846471,Hybrid
4,robustness,llm_full_html,4200,0,0,1.000000,1.000000,1.000000,4200,0,0,1.000000,1.000000,1.000000,4200,0,0,1.000000,1.000000,1.000000,Full-HTML LLM
5,robustness,original_fast,2394,0,1806,1.000000,0.570000,0.726115,2394,0,1806,1.000000,0.570000,0.726115,2394,0,1806,1.000000,0.570000,0.726115,Deterministic
6,synthetic,hybrid_original,316,0,104,1.000000,0.752381,0.858696,316,0,104,1.000000,0.752381,0.858696,316,0,104,1.000000,0.752381,0.858696,Hybrid
7,synthetic,llm_full_html,420,0,0,1.000000,1.000000,1.000000,420,0,0,1.000000,1.000000,1.000000,420,0,0,1.000000,1.000000,1.000000,Full-HTML LLM
8,synthetic,original_fast,252,0,168,1.000000,0.600000,0.750000,252,0,168,1.000000,0.600000,0.750000,252,0,168,1.000000,0.600000,0.750000,Deterministic


,variant,development_group,entity_TP,entity_FP,entity_FN,entity_P,entity_R,entity_F1,variant_label
0,hybrid_original,tailored,5460,271,338,0.952713,0.941704,0.947177,Hybrid
1,hybrid_original,untailored,391,4,128,0.989873,0.753372,0.855580,Hybrid
2,llm_full_html,tailored,5451,73,347,0.986785,0.940152,0.962904,Full-HTML LLM
3,llm_full_html,untailored,519,0,0,1.000000,1.000000,1.000000,Full-HTML LLM
4,original_fast,tailored,5437,269,361,0.952857,0.937737,0.945236,Deterministic
5,original_fast,untailored,362,4,157,0.989071,0.697495,0.818079,Deterministic


## 4. Figure export helpers

Every figure is saved at 300 dpi as PNG and also as vector PDF for paper production. Matplotlib's default color cycle is used deliberately; the notebook does not encode semantic meaning only by color.

In [23]:
def savefig(fig, stem: str):
    fig.tight_layout()
    png = FIGURE_DIR / f"{stem}.png"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("saved", png.name)


def ordered_present(values, preferred):
    values=list(dict.fromkeys([str(v) for v in values]))
    return [x for x in preferred if x in values] + [x for x in values if x not in preferred]


def grouped_bar(df, index_col, series_col, value_col, title, ylabel, stem, ylim=None):
    if df.empty or value_col not in df:
        return
    pivot=df.pivot_table(index=index_col, columns=series_col, values=value_col, aggfunc="first")
    pivot=pivot.reindex([x for x in DATASET_ORDER if x in pivot.index] + [x for x in pivot.index if x not in DATASET_ORDER])
    cols=ordered_present(pivot.columns, VARIANT_ORDER)
    pivot=pivot[cols]
    ax=pivot.rename(columns=DISPLAY_NAMES).plot(kind="bar", figsize=(9,5), rot=0)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel("")
    if ylim: ax.set_ylim(*ylim)
    ax.legend(title="")
    savefig(ax.figure, stem)


## 5. Overall quality and field-level quality

In [24]:
grouped_bar(overall, "dataset", "variant", "entity_F1", "Entity F1 by dataset and extraction strategy", "Entity F1", "01_entity_f1_by_dataset", (0,1.05))
grouped_bar(overall, "dataset", "variant", "entity_R", "Entity recall by dataset and extraction strategy", "Entity recall", "02_entity_recall_by_dataset", (0,1.05))
grouped_bar(overall, "dataset", "variant", "entity_P", "Entity precision by dataset and extraction strategy", "Entity precision", "03_entity_precision_by_dataset", (0,1.05))

# Precision-recall scatter. Use equal x/y scales so geometric distance is meaningful.
def precision_recall_scatter_window(xmin: float, stem: str, title_suffix: str):
    fig, ax = plt.subplots(figsize=(7,7))
    offsets=[(5,5),(5,-13),(-72,5),(-72,-13),(8,14),(-80,14),(8,-22),(-80,-22)]
    for i, (_, r) in enumerate(overall.iterrows()):
        if pd.isna(r.get("entity_P")) or pd.isna(r.get("entity_R")):
            continue
        ax.scatter(r["entity_R"], r["entity_P"], s=58)
        dx,dy=offsets[i % len(offsets)]
        ax.annotate(
            f"{DISPLAY_NAMES.get(r['variant'],r['variant'])}\n{r['dataset']}",
            (r["entity_R"],r["entity_P"]), xytext=(dx,dy),
            textcoords="offset points", fontsize=8, clip_on=True,
        )
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"Entity precision–recall trade-off {title_suffix}")
    ax.set_xlim(xmin,1.005); ax.set_ylim(xmin,1.005)
    ax.set_aspect("equal", adjustable="box"); ax.grid(alpha=.25)
    savefig(fig,stem)

precision_recall_scatter_window(0.50,"04_precision_recall_scatter","(0.50–1.00)")
precision_recall_scatter_window(0.90,"04b_precision_recall_scatter_zoom_090_100","(0.90–1.00 zoom)")

# Entity/name/price F1 facets as one grouped table plot per dataset.
field_rows=[]
for _, r in overall.iterrows():
    for field in ["entity","name","price"]:
        col=f"{field}_F1"
        if col in r and pd.notna(r[col]):
            field_rows.append({"dataset":r["dataset"],"variant":r["variant"],"field":field,"F1":r[col]})
field_df=pd.DataFrame(field_rows)
for dataset in DATASET_ORDER:
    d=field_df[field_df["dataset"]==dataset]
    if d.empty: continue
    piv=d.pivot(index="field",columns="variant",values="F1")
    cols=ordered_present(piv.columns,VARIANT_ORDER); piv=piv[cols]
    ax=piv.rename(columns=DISPLAY_NAMES).plot(kind="bar",figsize=(8,5),rot=0)
    ax.set_ylim(0,1.05); ax.set_ylabel("F1"); ax.set_xlabel(""); ax.set_title(f"Field-level F1 — {dataset}"); ax.legend(title="")
    savefig(ax.figure,f"05_field_f1_{dataset}")


saved 01_entity_f1_by_dataset.png
saved 02_entity_recall_by_dataset.png
saved 03_entity_precision_by_dataset.png
saved 04_precision_recall_scatter.png
saved 05_field_f1_synthetic.png
saved 05_field_f1_robustness.png
saved 05_field_f1_industrial.png


## 6. Tailored vs untailored domains and domain shift

This section is central to the paper's robustness claim. It reports micro metrics within each development group, macro-over-domain F1, and per-domain changes from deterministic extraction to the hybrid and full-HTML LLM arms.

In [25]:
if not dev_entity.empty:
    # Group-level F1/Recall/Precision.
    for metric in ["entity_F1","entity_R","entity_P"]:
        piv=dev_entity.pivot(index="development_group",columns="variant",values=metric)
        cols=ordered_present(piv.columns,VARIANT_ORDER); piv=piv[cols]
        ax=piv.rename(columns=DISPLAY_NAMES).plot(kind="bar",figsize=(8,5),rot=0)
        ax.set_ylim(0,1.05); ax.set_ylabel(metric.replace("entity_", "Entity ")); ax.set_xlabel("")
        ax.set_title(f"Industrial {metric.replace('entity_','').replace('_',' ')} by domain tailoring")
        ax.legend(title="")
        savefig(ax.figure,f"06_development_group_{metric.lower()}")

if not per_domain.empty:
    # Macro-over-domains summary — equal weight per domain.
    macro_domain=(per_domain.groupby(["variant","development_group"],dropna=False)
                  .agg(domain_macro_F1=("entity_F1","mean"), domain_median_F1=("entity_F1","median"), n_domains=("source","nunique"))
                  .reset_index())
    macro_domain.to_csv(TABLE_DIR / "domain_macro_metrics.csv",index=False)
    display(macro_domain)

    # Heatmap-like matrix of per-domain F1.
    mat=per_domain.pivot_table(index="source",columns="variant",values="entity_F1",aggfunc="first")
    cols=ordered_present(mat.columns,VARIANT_ORDER); mat=mat[cols]
    # Sort by deterministic F1 if present.
    if "original_fast" in mat.columns: mat=mat.sort_values("original_fast")
    fig,ax=plt.subplots(figsize=(8,max(6,0.32*len(mat))))
    im=ax.imshow(mat.to_numpy(),aspect="auto",vmin=0,vmax=1)
    ax.set_yticks(range(len(mat))); ax.set_yticklabels(mat.index,fontsize=8)
    ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels([DISPLAY_NAMES.get(c,c) for c in mat.columns],rotation=20,ha="right")
    ax.set_title("Industrial entity F1 by domain")
    fig.colorbar(im,ax=ax,label="Entity F1")
    savefig(fig,"07_per_domain_f1_heatmap")

    # Delta against deterministic.
    wide=per_domain.pivot_table(index=["source","development_group"],columns="variant",values="entity_F1",aggfunc="first").reset_index()
    for v in ["hybrid_original","llm_full_html"]:
        if v in wide and "original_fast" in wide:
            wide[f"delta_{v}_vs_deterministic"] = wide[v]-wide["original_fast"]
    wide.to_csv(TABLE_DIR / "per_domain_f1_deltas.csv",index=False)
    if "delta_hybrid_original_vs_deterministic" in wide:
        d=wide.sort_values("delta_hybrid_original_vs_deterministic")
        fig,ax=plt.subplots(figsize=(9,max(6,0.32*len(d))))
        ax.barh(d["source"],d["delta_hybrid_original_vs_deterministic"])
        ax.axvline(0,linewidth=1)
        ax.set_xlabel("Δ Entity F1 (Hybrid − Deterministic)"); ax.set_title("Hybrid gain/loss by industrial domain")
        savefig(fig,"08_hybrid_delta_f1_by_domain")

saved 06_development_group_entity_f1.png
saved 06_development_group_entity_r.png
saved 06_development_group_entity_p.png


,variant,development_group,domain_macro_F1,domain_median_F1,n_domains
0,hybrid_original,tailored,0.930444,0.999234,21
1,hybrid_original,untailored,0.608930,0.608930,2
2,llm_full_html,tailored,0.771896,0.999433,21
3,llm_full_html,untailored,1.000000,1.000000,2
4,original_fast,tailored,0.926605,0.999234,21
5,original_fast,untailored,0.428402,0.428402,2


saved 07_per_domain_f1_heatmap.png
saved 08_hybrid_delta_f1_by_domain.png


## 7. Error decomposition

In [26]:
# FP/FN counts by dataset and variant.
err=overall[["dataset","variant","entity_FP","entity_FN"]].copy()
for kind in ["entity_FP","entity_FN"]:
    piv=err.pivot(index="dataset",columns="variant",values=kind)
    piv=piv.reindex([d for d in DATASET_ORDER if d in piv.index])
    cols=ordered_present(piv.columns,VARIANT_ORDER); piv=piv[cols]
    ax=piv.rename(columns=DISPLAY_NAMES).plot(kind="bar",figsize=(9,5),rot=0)
    ax.set_ylabel(kind.replace("entity_","Entity ")); ax.set_xlabel(""); ax.set_title(f"{kind.replace('_',' ')} by dataset")
    ax.legend(title="")
    savefig(ax.figure,f"09_{kind.lower()}_counts")

# Error label taxonomy.
if "error_labels" in all_results.columns:
    records=[]
    for _,r in all_results[["dataset","variant","error_labels"]].dropna(subset=["error_labels"]).iterrows():
        raw=str(r["error_labels"]).strip()
        if not raw or raw.lower() in {"nan","none","[]"}: continue
        labels=[x.strip(" []'\"") for x in re.split(r"[,;|]",raw) if x.strip(" []'\"")]
        for label in labels:
            records.append({"dataset":r["dataset"],"variant":r["variant"],"error_label":label})
    error_labels=pd.DataFrame(records)
    if not error_labels.empty:
        counts=error_labels.groupby(["dataset","variant","error_label"]).size().rename("count").reset_index()
        counts.to_csv(TABLE_DIR / "error_label_counts.csv",index=False)
        top=counts.groupby("error_label")["count"].sum().nlargest(15).index
        d=counts[counts["error_label"].isin(top)].groupby(["error_label","variant"])["count"].sum().unstack(fill_value=0)
        d=d.loc[d.sum(axis=1).sort_values().index]
        cols=ordered_present(d.columns,VARIANT_ORDER); d=d[cols]
        ax=d.rename(columns=DISPLAY_NAMES).plot(kind="barh",figsize=(10,7))
        ax.set_xlabel("Count"); ax.set_ylabel(""); ax.set_title("Most frequent extraction error labels")
        savefig(ax.figure,"10_error_label_taxonomy")

# Name similarity and price relative error distributions.
if "name_similarity" in all_results:
    fig,ax=plt.subplots(figsize=(9,5))
    for v in VARIANT_ORDER:
        x=pd.to_numeric(all_results.loc[(all_results["variant"]==v)&all_results["gt_id"].notna(),"name_similarity"],errors="coerce").dropna()
        if len(x): ax.hist(x,bins=np.linspace(0,1,31),histtype="step",linewidth=1.5,label=DISPLAY_NAMES[v],density=True)
    ax.set_xlabel("Name similarity"); ax.set_ylabel("Density"); ax.set_title("Matched-name similarity distribution"); ax.legend()
    savefig(fig,"11_name_similarity_distribution")

if "price_error_relative_absolute" in all_results:
    fig,ax=plt.subplots(figsize=(9,5))
    for v in VARIANT_ORDER:
        x=pd.to_numeric(all_results.loc[all_results["variant"]==v,"price_error_relative_absolute"],errors="coerce").dropna()
        x=x[(x>=0)&np.isfinite(x)]
        if len(x):
            cap=max(0.05,float(x.quantile(.99)))
            ax.hist(x.clip(upper=cap),bins=40,histtype="step",linewidth=1.5,label=DISPLAY_NAMES[v],density=True)
    ax.set_xlabel("Absolute relative price error (99th percentile clipped per arm)"); ax.set_ylabel("Density")
    ax.set_title("Price error distribution"); ax.legend()
    savefig(fig,"12_price_relative_error_distribution")

saved 09_entity_fp_counts.png
saved 09_entity_fn_counts.png
saved 10_error_label_taxonomy.png
saved 11_name_similarity_distribution.png
saved 12_price_relative_error_distribution.png


## 8. Robustness mutation analysis

If the robustness dataset includes mutation metadata, metrics are stratified by mutation axis. This can reveal whether failures are driven by DOM depth, attribute perturbations, wrappers, text noise, or other synthetic stressors rather than by a single aggregate score.

In [27]:
mutation_col = "page_meta__mutation_axis" if "page_meta__mutation_axis" in pages.columns else None
if mutation_col and pages[mutation_col].notna().any():
    r=pages[(pages["dataset"].astype(str)=="robustness") & pages[mutation_col].notna()].copy()
    mut=aggregate_prf(r,["variant",mutation_col],"entity")
    mut.to_csv(TABLE_DIR / "robustness_by_mutation_axis.csv",index=False)
    piv=mut.pivot(index=mutation_col,columns="variant",values="entity_F1")
    cols=ordered_present(piv.columns,VARIANT_ORDER); piv=piv[cols]
    ax=piv.rename(columns=DISPLAY_NAMES).plot(kind="bar",figsize=(11,6),rot=35)
    ax.set_ylim(0,1.05); ax.set_ylabel("Entity F1"); ax.set_xlabel("Mutation axis"); ax.set_title("Robustness F1 by mutation axis"); ax.legend(title="")
    savefig(ax.figure,"13_robustness_f1_by_mutation_axis")
else:
    print("Mutation-axis metadata unavailable; skipping robustness stratification.")

saved 13_robustness_f1_by_mutation_axis.png


## 9. Page-level difficulty, latency, and structural signals

In [28]:
# Page-level F1 distribution.
if "entity_F1" in pages:
    data=[]; labels=[]
    for v in VARIANT_ORDER:
        x=pd.to_numeric(pages.loc[pages["variant"]==v,"entity_F1"],errors="coerce").dropna().to_numpy()
        if len(x): data.append(x); labels.append(DISPLAY_NAMES[v])
    if data:
        fig,ax=plt.subplots(figsize=(8,5)); ax.boxplot(data,tick_labels=labels,showfliers=False)
        ax.set_ylim(0,1.05); ax.set_ylabel("Page-level entity F1"); ax.set_title("Distribution of page-level extraction quality")
        savefig(fig,"14_page_level_f1_distribution")

# Local timing summaries exported in Pages. These are useful for deterministic
# extraction diagnostics. Provider/network LLM latency is loaded separately below.
latency_cols=[c for c in ["extract_ms","shared_extraction_ms","html_parse_ms","html_read_ms","query_filter_ms"] if c in pages.columns]
latency_rows=[]
for c in latency_cols:
    for (dataset,variant),g in pages.groupby(["dataset","variant"]):
        x=pd.to_numeric(g[c],errors="coerce").dropna()
        if len(x):
            latency_rows.append({
                "dataset":dataset,"variant":variant,"metric":c,"n":len(x),
                "min_ms":x.min(),"mean_ms":x.mean(),"median_ms":x.median(),
                "p95_ms":x.quantile(.95),"max_ms":x.max(),"std_ms":x.std(ddof=1),
            })
latency=pd.DataFrame(latency_rows)
if not latency.empty:
    latency.to_csv(TABLE_DIR / "latency_summary.csv",index=False)
    display(latency)

# Candidate count versus misses for arms that expose deterministic structural diagnostics.
if {"candidate_count","entity_FN"}.issubset(pages.columns):
    d=pages.copy()
    d["candidate_count"]=pd.to_numeric(d["candidate_count"],errors="coerce")
    d["entity_FN"]=pd.to_numeric(d["entity_FN"],errors="coerce")
    d=d.dropna(subset=["candidate_count","entity_FN"])
    if not d.empty:
        fig,ax=plt.subplots(figsize=(8,5))
        for v in VARIANT_ORDER:
            x=d[d["variant"]==v]
            if len(x): ax.scatter(x["candidate_count"],x["entity_FN"],alpha=.35,s=18,label=DISPLAY_NAMES[v])
        ax.set_xlabel("Candidate count"); ax.set_ylabel("False negatives on page"); ax.set_title("Structural candidate count vs missed entities"); ax.legend()
        savefig(fig,"15_candidate_count_vs_false_negatives")

# HTML/page size bins if metadata exists.
size_col="page_meta__char_len" if "page_meta__char_len" in pages.columns else None
if size_col and pages[size_col].notna().any():
    d=pages.copy(); d[size_col]=pd.to_numeric(d[size_col],errors="coerce")
    d=d[(d["dataset"].astype(str)=="industrial") & d[size_col].notna() & (d[size_col]>0)].copy()
    if len(d):
        d["size_bin"] = pd.qcut(d[size_col], q=min(5,d[size_col].nunique()), duplicates="drop")

        def compact_chars(value):
            value=float(value)
            if abs(value) >= 1_000_000:
                return f"{value/1_000_000:.1f}M"
            if abs(value) >= 1_000:
                return f"{value/1_000:.0f}k"
            return f"{value:.0f}"

        interval_order=list(d["size_bin"].cat.categories)
        label_map={iv:f"{compact_chars(iv.left)}–{compact_chars(iv.right)}" for iv in interval_order}
        label_order=[label_map[iv] for iv in interval_order]
        d["size_bin_label"]=d["size_bin"].map(label_map).astype(str)
        s=aggregate_prf(d,["variant","size_bin_label"],"entity")
        s.to_csv(TABLE_DIR / "industrial_by_html_size_bin.csv",index=False)
        piv=s.pivot(index="size_bin_label",columns="variant",values="entity_F1").reindex(label_order)
        cols=ordered_present(piv.columns,VARIANT_ORDER); piv=piv[cols]
        ax=piv.rename(columns=DISPLAY_NAMES).plot(marker="o",figsize=(10,5))
        ax.set_ylim(0,1.05); ax.set_ylabel("Entity F1"); ax.set_xlabel("HTML length (characters, rounded)"); ax.set_title("Industrial quality vs HTML size")
        ax.tick_params(axis="x", rotation=0)
        savefig(ax.figure,"16_industrial_f1_by_html_size")


saved 14_page_level_f1_distribution.png


,dataset,variant,metric,n,median_ms,p95_ms,mean_ms
0,industrial,llm_full_html,extract_ms,99,54675.46120,106435.702970,53747.336219
1,industrial,original_fast,extract_ms,239,1652.26650,18813.011430,4516.975480
2,robustness,original_fast,extract_ms,200,26.32225,50.763105,26.232447
3,synthetic,original_fast,extract_ms,20,76.03395,183.470170,87.175425
4,industrial,original_fast,shared_extraction_ms,239,1652.26650,18813.011430,4516.975480
5,robustness,original_fast,shared_extraction_ms,200,26.32225,50.763105,26.232447
6,synthetic,original_fast,shared_extraction_ms,20,76.03395,183.470170,87.175425
7,industrial,original_fast,html_read_ms,239,12.20630,15.613920,8.980117
8,robustness,original_fast,html_read_ms,200,0.28805,0.428120,0.305909
9,synthetic,original_fast,html_read_ms,20,0.48020,3.648030,0.956625


saved 15_candidate_count_vs_false_negatives.png
saved 16_industrial_f1_by_html_size.png


## 10. Hybrid routing and cache-efficiency diagnostics

These analyses are conditional on the diagnostic columns exported by the hybrid arm. They quantify how often the LLM was invoked, how many calls were replayed from cache, and which page states remain difficult after coverage-aware routing.

In [29]:
hp=pages[pages["variant"]=="hybrid_original"].copy()
summary=[]
if not hp.empty:
    for dataset,g in hp.groupby("dataset"):
        row={"dataset":dataset,"pages":len(g)}
        for c in ["llm_attempts","new_llm_attempts","rejected_count","candidate_count","returned_count"]:
            if c in g:
                x=pd.to_numeric(g[c],errors="coerce").fillna(0)
                row[f"{c}_sum"]=x.sum(); row[f"{c}_mean"]=x.mean()
        replay_cols=[c for c in g.columns if str(c).startswith("replay_cached")]
        if replay_cols:
            replay=pd.concat([normalize_bool_series(g[c]) for c in replay_cols],axis=1).max(axis=1)
            row["replay_cached_pages"]=int(replay.sum())
            row["replay_cached_rate"]=float(replay.mean())
        summary.append(row)
hybrid_routing=pd.DataFrame(summary)
if not hybrid_routing.empty:
    hybrid_routing.to_csv(TABLE_DIR / "hybrid_routing_summary.csv",index=False)
    display(hybrid_routing)

for col in ["extraction_status","original_status","error_type"]:
    if col in hp.columns and hp[col].notna().any():
        c=hp.groupby(["dataset",col]).size().rename("pages").reset_index()
        c.to_csv(TABLE_DIR / f"hybrid_{col}_counts.csv",index=False)

,dataset,pages,llm_attempts_sum,llm_attempts_mean,new_llm_attempts_sum,new_llm_attempts_mean,rejected_count_sum,rejected_count_mean,candidate_count_sum,candidate_count_mean,returned_count_sum,returned_count_mean,replay_cached_pages,replay_cached_rate
0,industrial,240,164.0,0.683333,0.0,0.0,0.0,0.0,6090.0,25.375,6126.0,25.525,164,0.683333
1,robustness,200,86.0,0.430000,0.0,0.0,0.0,0.0,2394.0,11.970,3082.0,15.410,86,0.430000
2,synthetic,20,8.0,0.400000,0.0,0.0,0.0,0.0,252.0,12.600,316.0,15.800,8,0.400000


## 11. Cost–quality Pareto analysis

The cost table uses the reported billing figures configured at the top of the notebook. Quality remains benchmark-derived. This separation makes the provenance of the economic claim explicit.

In [30]:
cost=COST_OVERRIDES.copy()
ind=overall[overall["dataset"].astype(str)=="industrial"][["variant","entity_P","entity_R","entity_F1"]].copy()
cost_quality=cost.merge(ind,on="variant",how="left")
# Number of evaluated industrial pages from Pages rather than hard-coding 239/240.
npages=pages[pages["dataset"].astype(str)=="industrial"].groupby("variant")["page_id"].nunique().rename("industrial_pages")
cost_quality=cost_quality.merge(npages,on="variant",how="left")
cost_quality["cost_usd"]=cost_quality["cost_rub"] / RUB_PER_USD
cost_quality["rub_per_usd_experiment"]=RUB_PER_USD
cost_quality["tokens_per_page"]=cost_quality["total_tokens"] / cost_quality["industrial_pages"].replace(0,np.nan)
cost_quality["rub_per_page"]=cost_quality["cost_rub"] / cost_quality["industrial_pages"].replace(0,np.nan)
cost_quality["usd_per_page"]=cost_quality["cost_usd"] / cost_quality["industrial_pages"].replace(0,np.nan)
cost_quality["F1_per_million_tokens"]=np.where(cost_quality["total_tokens"]>0,cost_quality["entity_F1"]/(cost_quality["total_tokens"]/1e6),np.nan)
cost_quality.to_csv(TABLE_DIR / "cost_quality_summary.csv",index=False)
display(cost_quality)

fig,ax=plt.subplots(figsize=(8,6))
for _,r in cost_quality.dropna(subset=["entity_F1","cost_usd"]).iterrows():
    ax.scatter(r["cost_usd"],r["entity_F1"],s=80)
    label=f"{DISPLAY_NAMES.get(r['variant'],r['variant'])}\n${r['cost_usd']:.2f}"
    ax.annotate(label,(r["cost_usd"],r["entity_F1"]),xytext=(5,5),textcoords="offset points")
ax.set_xlabel(f"LLM cost (USD; {RUB_PER_USD:.1f} RUB/USD)"); ax.set_ylabel("Industrial entity F1"); ax.set_title("Cost–quality trade-off")
ax.set_ylim(max(0,float(cost_quality["entity_F1"].min())-.05),1.01); ax.grid(alpha=.25)
savefig(fig,"17_cost_quality_pareto")

fig,ax=plt.subplots(figsize=(8,6))
for _,r in cost_quality.dropna(subset=["entity_F1","total_tokens"]).iterrows():
    tokens_m=r["total_tokens"]/1e6
    ax.scatter(tokens_m,r["entity_F1"],s=80)
    label=f"{DISPLAY_NAMES.get(r['variant'],r['variant'])}\n{tokens_m:.3g}M"
    ax.annotate(label,(tokens_m,r["entity_F1"]),xytext=(5,5),textcoords="offset points")
ax.set_xlabel("Reported LLM tokens (millions)"); ax.set_ylabel("Industrial entity F1"); ax.set_title("Token–quality trade-off"); ax.grid(alpha=.25)
savefig(fig,"18_token_quality_pareto")

# Savings relative to full-HTML LLM.
try:
    full=cost_quality.set_index("variant").loc["llm_full_html"]
    hyb=cost_quality.set_index("variant").loc["hybrid_original"]
    savings=pd.DataFrame([{
        "token_saving_fraction":1-hyb.total_tokens/full.total_tokens if full.total_tokens else np.nan,
        "cost_saving_fraction":1-hyb.cost_usd/full.cost_usd if full.cost_usd else np.nan,
        "full_llm_cost_usd":full.cost_usd,
        "hybrid_cost_usd":hyb.cost_usd,
        "rub_per_usd_experiment":RUB_PER_USD,
        "industrial_F1_gap":hyb.entity_F1-full.entity_F1,
        "industrial_recall_gap":hyb.entity_R-full.entity_R,
        "industrial_precision_gap":hyb.entity_P-full.entity_P,
    }])
    savings.to_csv(TABLE_DIR / "hybrid_vs_full_llm_efficiency.csv",index=False)
    display(savings)
except KeyError:
    pass


,variant,total_tokens,cost_rub,cost_source,entity_P,entity_R,entity_F1,industrial_pages,tokens_per_page,rub_per_page,F1_per_million_tokens
0,original_fast,0,0.0,deterministic; no LLM inference,0.955040,0.917999,0.936153,240,0.0,0.000000,NaN
1,llm_full_html,60000000,1000.0,reported rounded provider billing,0.987920,0.945069,0.966019,240,250000.0,4.166667,0.016100
2,hybrid_original,3000000,60.5,reported rounded provider billing,0.955109,0.926231,0.940448,240,12500.0,0.252083,0.313483


saved 17_cost_quality_pareto.png
saved 18_token_quality_pareto.png


,token_saving_fraction,cost_saving_fraction,industrial_F1_gap,industrial_recall_gap,industrial_precision_gap
0,0.95,0.9395,-0.025571,-0.018838,-0.032811


## 12. Multivariate efficiency, domain recovery, and latency diagnostics

These figures emphasize multivariate relationships rather than additional histograms. Latency from saved LLM checkpoints is provider/network request latency, not a controlled end-to-end systems benchmark.


In [ ]:
# ---- Bubble view: cost, quality, and token volume in one figure. ----
if not cost_quality.empty:
    d=cost_quality.dropna(subset=["entity_F1","cost_usd","total_tokens"]).copy()
    if not d.empty:
        max_tokens=max(float(d["total_tokens"].max()),1.0)
        sizes=120 + 1100*np.sqrt(d["total_tokens"].astype(float)/max_tokens)
        fig,ax=plt.subplots(figsize=(8,6))
        for (_,r),size in zip(d.iterrows(),sizes):
            ax.scatter(r["cost_usd"],r["entity_F1"],s=size,alpha=.65)
            ax.annotate(
                f"{DISPLAY_NAMES.get(r['variant'],r['variant'])}\n{r['total_tokens']/1e6:.2f}M tokens",
                (r["cost_usd"],r["entity_F1"]),xytext=(6,6),textcoords="offset points",fontsize=8,
            )
        ax.set_xlabel(f"LLM cost (USD; {RUB_PER_USD:.1f} RUB/USD)")
        ax.set_ylabel("Industrial entity F1")
        ax.set_title("Efficiency bubble: quality, cost, and token volume")
        ax.grid(alpha=.25)
        savefig(fig,"21_cost_quality_token_bubble")

# ---- Per-domain bubble: Hybrid vs deterministic. Bubble area ~= GT entities. ----
if not per_domain.empty:
    d=per_domain.copy()
    d["gt_entities"]=pd.to_numeric(d["entity_TP"],errors="coerce").fillna(0)+pd.to_numeric(d["entity_FN"],errors="coerce").fillna(0)
    wide=d.pivot_table(index=["source","development_group"],columns="variant",values="entity_F1",aggfunc="first")
    gt=d.groupby(["source","development_group"],dropna=False)["gt_entities"].max()
    wide=wide.join(gt).reset_index()
    if {"original_fast","hybrid_original"}.issubset(wide.columns):
        q=wide.dropna(subset=["original_fast","hybrid_original"]).copy()
        if not q.empty:
            sizes=50+650*np.sqrt(q["gt_entities"].clip(lower=1)/max(float(q["gt_entities"].max()),1.0))
            fig,ax=plt.subplots(figsize=(7.5,7.5))
            for group in ["tailored","untailored"]:
                m=q["development_group"].astype(str)==group
                if m.any():
                    ax.scatter(q.loc[m,"original_fast"],q.loc[m,"hybrid_original"],s=sizes[m.to_numpy()],alpha=.60,label=group)
            lo=max(0,min(float(q["original_fast"].min()),float(q["hybrid_original"].min()))-.04)
            ax.plot([lo,1],[lo,1],linestyle="--",linewidth=1)
            q["delta"]=q["hybrid_original"]-q["original_fast"]
            for _,r in q.reindex(q["delta"].abs().sort_values(ascending=False).index).head(6).iterrows():
                ax.annotate(str(r["source"]),(r["original_fast"],r["hybrid_original"]),xytext=(4,4),textcoords="offset points",fontsize=7)
            ax.set_xlim(lo,1.01); ax.set_ylim(lo,1.01); ax.set_aspect("equal",adjustable="box")
            ax.set_xlabel("Deterministic domain F1"); ax.set_ylabel("Hybrid domain F1")
            ax.set_title("Per-domain quality shift (bubble area ≈ GT entities)"); ax.legend(title="Domain group")
            ax.grid(alpha=.2)
            savefig(fig,"22_domain_hybrid_vs_deterministic_bubble")

# ---- Domain-level precision/recall change caused by hybrid routing. ----
if not per_domain.empty:
    p=per_domain.pivot_table(index=["source","development_group"],columns="variant",values="entity_P",aggfunc="first")
    r=per_domain.pivot_table(index=["source","development_group"],columns="variant",values="entity_R",aggfunc="first")
    g=per_domain.assign(gt_entities=per_domain["entity_TP"]+per_domain["entity_FN"]).groupby(["source","development_group"],dropna=False)["gt_entities"].max()
    if {"original_fast","hybrid_original"}.issubset(p.columns) and {"original_fast","hybrid_original"}.issubset(r.columns):
        q=pd.DataFrame({
            "delta_precision":p["hybrid_original"]-p["original_fast"],
            "delta_recall":r["hybrid_original"]-r["original_fast"],
            "gt_entities":g,
        }).dropna().reset_index()
        if not q.empty:
            sizes=50+650*np.sqrt(q["gt_entities"].clip(lower=1)/max(float(q["gt_entities"].max()),1.0))
            fig,ax=plt.subplots(figsize=(8,6))
            for group in ["tailored","untailored"]:
                m=q["development_group"].astype(str)==group
                if m.any(): ax.scatter(q.loc[m,"delta_recall"],q.loc[m,"delta_precision"],s=sizes[m.to_numpy()],alpha=.60,label=group)
            ax.axhline(0,linewidth=1); ax.axvline(0,linewidth=1)
            ax.set_xlabel("Δ Recall (Hybrid − Deterministic)"); ax.set_ylabel("Δ Precision (Hybrid − Deterministic)")
            ax.set_title("Where hybrid gains come from by domain")
            ax.legend(title="Domain group"); ax.grid(alpha=.2)
            savefig(fig,"23_domain_precision_recall_delta_bubble")

# ---- Paired industrial page quality: deterministic vs hybrid, bubble ~= GT products. ----
def _page_metric_frame(variant, dataset="industrial"):
    q=pages[(pages["variant"]==variant)&(pages["dataset"].astype(str)==dataset)].copy()
    keep=[c for c in ["page_id","entity_F1","gt_positive_count","entity_TP","entity_FP","entity_FN"] if c in q.columns]
    q=q[keep].copy()
    for c in ["entity_F1","gt_positive_count","entity_TP","entity_FP","entity_FN"]:
        if c in q: q[c]=pd.to_numeric(q[c],errors="coerce")
    return q.drop_duplicates("page_id")

pd_det=_page_metric_frame("original_fast")
pd_hyb=_page_metric_frame("hybrid_original")
if not pd_det.empty and not pd_hyb.empty and "entity_F1" in pd_det and "entity_F1" in pd_hyb:
    q=pd_det.merge(pd_hyb,on="page_id",suffixes=("_det","_hyb")).dropna(subset=["entity_F1_det","entity_F1_hyb"])
    if not q.empty:
        gt=q.get("gt_positive_count_det",pd.Series(1,index=q.index)).fillna(0).clip(lower=1)
        sizes=18+180*np.sqrt(gt/max(float(gt.max()),1.0))
        fig,ax=plt.subplots(figsize=(7.5,7.5))
        ax.scatter(q["entity_F1_det"],q["entity_F1_hyb"],s=sizes,alpha=.35)
        ax.plot([0,1],[0,1],linestyle="--",linewidth=1)
        ax.set_xlim(-.01,1.01); ax.set_ylim(-.01,1.01); ax.set_aspect("equal",adjustable="box")
        ax.set_xlabel("Deterministic page F1"); ax.set_ylabel("Hybrid page F1")
        ax.set_title("Paired industrial page quality (bubble area ≈ GT products)")
        ax.grid(alpha=.2)
        savefig(fig,"24_page_hybrid_vs_deterministic_bubble")

# ---- HTML size vs page quality, a direct page-level view complementary to Fig. 16. ----
if size_col and size_col in pages.columns:
    q=pages[pages["dataset"].astype(str)=="industrial"].copy()
    q[size_col]=pd.to_numeric(q[size_col],errors="coerce")
    q["entity_F1"]=pd.to_numeric(q.get("entity_F1"),errors="coerce")
    q=q.dropna(subset=[size_col,"entity_F1"])
    if not q.empty:
        fig,ax=plt.subplots(figsize=(9,6))
        for v in VARIANT_ORDER:
            z=q[q["variant"]==v]
            if len(z): ax.scatter(z[size_col],z["entity_F1"],s=18,alpha=.28,label=DISPLAY_NAMES[v])
        ax.set_xscale("log"); ax.set_ylim(-.02,1.02)
        ax.set_xlabel("HTML length (characters, log scale)"); ax.set_ylabel("Page-level entity F1")
        ax.set_title("Industrial page quality vs HTML size"); ax.legend(); ax.grid(alpha=.2)
        savefig(fig,"25_html_size_vs_page_f1_scatter")

# ---- Candidate count vs page F1: bubble area ~= ground-truth product count. ----
if {"candidate_count","entity_F1"}.issubset(pages.columns):
    q=pages[pages["dataset"].astype(str)=="industrial"].copy()
    q["candidate_count"]=pd.to_numeric(q["candidate_count"],errors="coerce")
    q["entity_F1"]=pd.to_numeric(q["entity_F1"],errors="coerce")
    q=q.dropna(subset=["candidate_count","entity_F1"])
    if not q.empty:
        gt=pd.to_numeric(q.get("gt_positive_count",1),errors="coerce").fillna(1).clip(lower=1)
        sizes=14+120*np.sqrt(gt/max(float(gt.max()),1.0))
        fig,ax=plt.subplots(figsize=(9,6))
        for v in VARIANT_ORDER:
            m=q["variant"]==v
            if m.any(): ax.scatter(q.loc[m,"candidate_count"],q.loc[m,"entity_F1"],s=sizes[m.to_numpy()],alpha=.28,label=DISPLAY_NAMES[v])
        ax.set_xlabel("Structural candidate count"); ax.set_ylabel("Page-level entity F1")
        ax.set_title("Candidate density vs extraction quality"); ax.legend(); ax.grid(alpha=.2)
        savefig(fig,"26_candidate_count_vs_page_f1_bubble")

# ---- Recover saved per-request LLM latency from immutable result.json checkpoints. ----
def scan_saved_llm_latency(root: Path, variant: str) -> pd.DataFrame:
    rows=[]
    if not root.exists(): return pd.DataFrame()
    for path in root.rglob("result.json"):
        try:
            obj=json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        dataset=obj.get("dataset"); page_id=obj.get("page_id")
        attempts=obj.get("attempts") or []
        vals=[]
        for a in attempts:
            try:
                value=float(a.get("latency_ms"))
            except (TypeError,ValueError):
                continue
            if np.isfinite(value) and value>=0: vals.append(value)
        if dataset is None or page_id is None or not vals: continue
        rows.append({
            "dataset":str(dataset),"variant":variant,"page_id":str(page_id),
            "checkpoint":str(path.relative_to(root)),"http_attempts":len(vals),
            "api_latency_ms":float(sum(vals)),"api_attempt_mean_ms":float(np.mean(vals)),
            "api_attempt_min_ms":float(np.min(vals)),"api_attempt_max_ms":float(np.max(vals)),
        })
    if not rows: return pd.DataFrame()
    frame=pd.DataFrame(rows)
    # Hybrid can have several bounded stages for one page; page API latency is their sum.
    return (frame.groupby(["dataset","variant","page_id"],as_index=False)
            .agg(api_latency_ms=("api_latency_ms","sum"),http_attempts=("http_attempts","sum"),
                 api_attempt_min_ms=("api_attempt_min_ms","min"),api_attempt_max_ms=("api_attempt_max_ms","max")))

api_latency_parts=[]
for variant in ["llm_full_html","hybrid_original"]:
    frame=scan_saved_llm_latency(RESULT_ROOTS[variant],variant)
    if not frame.empty: api_latency_parts.append(frame)
api_latency=pd.concat(api_latency_parts,ignore_index=True) if api_latency_parts else pd.DataFrame()

if not api_latency.empty:
    api_latency.to_csv(TABLE_DIR / "saved_llm_api_latency_by_page.csv",index=False)
    lat_summary=(api_latency.groupby(["dataset","variant"])["api_latency_ms"]
                 .agg(n="count",min_ms="min",mean_ms="mean",median_ms="median",max_ms="max")
                 .reset_index())
    lat_summary["p95_ms"]=api_latency.groupby(["dataset","variant"])["api_latency_ms"].quantile(.95).to_numpy()
    lat_summary.to_csv(TABLE_DIR / "saved_llm_api_latency_summary.csv",index=False)
    display(lat_summary)

    # Min--mean--max range plot for the industrial LLM-bearing arms.
    q=lat_summary[lat_summary["dataset"].astype(str)=="industrial"].copy()
    if not q.empty:
        q["label"]=q["variant"].map(DISPLAY_NAMES).fillna(q["variant"])
        y=np.arange(len(q)); mean=q["mean_ms"].to_numpy()/1000
        low=(q["mean_ms"]-q["min_ms"]).to_numpy()/1000
        high=(q["max_ms"]-q["mean_ms"]).to_numpy()/1000
        fig,ax=plt.subplots(figsize=(9,4.8))
        ax.errorbar(mean,y,xerr=np.vstack([low,high]),fmt="o",capsize=5)
        ax.set_yticks(y); ax.set_yticklabels(q["label"])
        ax.set_xlabel("Saved LLM API latency per page (seconds): min — mean — max")
        ax.set_title("Industrial recorded inference latency range")
        ax.grid(axis="x",alpha=.25)
        savefig(fig,"27_llm_api_latency_min_mean_max")

    # Latency ECDF: more informative than another histogram.
    q=api_latency[api_latency["dataset"].astype(str)=="industrial"].copy()
    if not q.empty:
        fig,ax=plt.subplots(figsize=(9,5.5))
        for v in ["hybrid_original","llm_full_html"]:
            x=np.sort(pd.to_numeric(q.loc[q["variant"]==v,"api_latency_ms"],errors="coerce").dropna().to_numpy()/1000)
            if len(x): ax.step(x,np.arange(1,len(x)+1)/len(x),where="post",label=DISPLAY_NAMES[v])
        ax.set_xlabel("Saved LLM API latency per page (seconds)"); ax.set_ylabel("Empirical cumulative probability")
        ax.set_title("Industrial LLM latency ECDF"); ax.legend(); ax.grid(alpha=.2)
        savefig(fig,"28_llm_api_latency_ecdf")

    # Accuracy vs recorded page latency. Horizontal ranges are min/max; points are means.
    q=lat_summary[lat_summary["dataset"].astype(str)=="industrial"].merge(
        overall[overall["dataset"].astype(str)=="industrial"][["variant","entity_F1"]],on="variant",how="left")
    if not q.empty:
        fig,ax=plt.subplots(figsize=(8,6))
        for _,r in q.dropna(subset=["entity_F1","mean_ms"]).iterrows():
            mean=r["mean_ms"]/1000; lo=(r["mean_ms"]-r["min_ms"])/1000; hi=(r["max_ms"]-r["mean_ms"])/1000
            ax.errorbar(mean,r["entity_F1"],xerr=np.array([[lo],[hi]]),fmt="o",capsize=4)
            ax.annotate(DISPLAY_NAMES.get(r["variant"],r["variant"]),(mean,r["entity_F1"]),xytext=(5,5),textcoords="offset points")
        ax.set_xlabel("Mean saved LLM API latency per page (seconds; whiskers min–max)")
        ax.set_ylabel("Industrial entity F1")
        ax.set_title("Recorded inference latency vs extraction accuracy")
        ax.grid(alpha=.25)
        savefig(fig,"29_llm_latency_vs_accuracy")
else:
    print("No saved result.json request latencies found; LLM latency figures 27–29 skipped.")

# Deterministic local extraction timing with accuracy. This timing is a different
# measurement basis from HTTP inference latency, so it is plotted separately.
if not latency.empty:
    q=latency[(latency["dataset"].astype(str)=="industrial") & (latency["metric"]=="extract_ms")].copy()
    q=q.merge(overall[overall["dataset"].astype(str)=="industrial"][["variant","entity_F1"]],on="variant",how="left")
    if not q.empty:
        fig,ax=plt.subplots(figsize=(8,5.5))
        for _,r in q.dropna(subset=["mean_ms","entity_F1"]).iterrows():
            mean=r["mean_ms"]; lo=mean-r["min_ms"]; hi=r["max_ms"]-mean
            ax.errorbar(mean,r["entity_F1"],xerr=np.array([[lo],[hi]]),fmt="o",capsize=4)
            ax.annotate(DISPLAY_NAMES.get(r["variant"],r["variant"]),(mean,r["entity_F1"]),xytext=(5,5),textcoords="offset points")
        ax.set_xlabel("Local extraction time per page (ms; mean with min–max range)")
        ax.set_ylabel("Industrial entity F1")
        ax.set_title("Local extraction latency vs accuracy")
        ax.grid(alpha=.25)
        savefig(fig,"30_local_extraction_latency_vs_accuracy")


## 12. Bootstrap confidence intervals and paired page-level effects

Bootstrap resampling is done at the **page level**, not at the entity-row level, to reduce pseudo-replication from pages containing many products. The industrial development-group CIs are therefore suitable as uncertainty estimates around the observed micro-F1. Paired effects are computed only on page IDs shared by both arms.

In [31]:
rng=np.random.default_rng(BOOTSTRAP_SEED)

def micro_f1_from_pages(df):
    return prf(df["entity_TP"].sum(),df["entity_FP"].sum(),df["entity_FN"].sum())[2]

def bootstrap_page_micro_f1(df,n=BOOTSTRAP_N):
    d=df[["page_id","entity_TP","entity_FP","entity_FN"]].copy()
    d=d.groupby("page_id",as_index=False)[["entity_TP","entity_FP","entity_FN"]].sum()
    if len(d)<2: return (np.nan,np.nan,np.nan)
    vals=np.empty(n,float)
    arr=d[["entity_TP","entity_FP","entity_FN"]].to_numpy(float)
    for i in range(n):
        sample=arr[rng.integers(0,len(arr),len(arr))]
        vals[i]=prf(*sample.sum(axis=0))[2]
    return float(np.mean(vals)),float(np.quantile(vals,.025)),float(np.quantile(vals,.975))

ci_rows=[]
for (dataset,variant),g in pages.groupby(["dataset","variant"]):
    mean,lo,hi=bootstrap_page_micro_f1(g)
    ci_rows.append({"dataset":dataset,"variant":variant,"development_group":"all","bootstrap_mean_F1":mean,"ci95_low":lo,"ci95_high":hi,"n_pages":g["page_id"].nunique()})
if "development_group" in pages.columns:
    d=pages[(pages["dataset"].astype(str)=="industrial") & pages["development_group"].isin(["tailored","untailored"])]
    for (variant,group),g in d.groupby(["variant","development_group"]):
        mean,lo,hi=bootstrap_page_micro_f1(g)
        ci_rows.append({"dataset":"industrial","variant":variant,"development_group":group,"bootstrap_mean_F1":mean,"ci95_low":lo,"ci95_high":hi,"n_pages":g["page_id"].nunique()})
bootstrap_ci=pd.DataFrame(ci_rows)
bootstrap_ci.to_csv(TABLE_DIR / "bootstrap_page_f1_ci.csv",index=False)
display(bootstrap_ci)

# Plot industrial CIs.
d=bootstrap_ci[(bootstrap_ci["dataset"]=="industrial") & (bootstrap_ci["development_group"]=="all")].copy()
if not d.empty:
    d["label"]=d["variant"].map(DISPLAY_NAMES)
    x=np.arange(len(d)); y=d["bootstrap_mean_F1"].to_numpy(); err=np.vstack([y-d["ci95_low"].to_numpy(),d["ci95_high"].to_numpy()-y])
    fig,ax=plt.subplots(figsize=(8,5)); ax.errorbar(x,y,yerr=err,fmt="o",capsize=5)
    ax.set_xticks(x); ax.set_xticklabels(d["label"]); ax.set_ylim(max(0,float(d["ci95_low"].min())-.03),1.01)
    ax.set_ylabel("Entity F1"); ax.set_title("Industrial entity F1 with page-bootstrap 95% CI")
    savefig(fig,"19_industrial_f1_bootstrap_ci")


def paired_page_delta(variant_a, variant_b, dataset="industrial", n=BOOTSTRAP_N):
    a=pages[(pages["variant"]==variant_a)&(pages["dataset"].astype(str)==dataset)][["page_id","entity_TP","entity_FP","entity_FN"]].copy()
    b=pages[(pages["variant"]==variant_b)&(pages["dataset"].astype(str)==dataset)][["page_id","entity_TP","entity_FP","entity_FN"]].copy()
    for d in (a,b):
        d["page_f1"]=[prf(tp,fp,fn)[2] for tp,fp,fn in d[["entity_TP","entity_FP","entity_FN"]].to_numpy()]
    m=a[["page_id","page_f1"]].merge(b[["page_id","page_f1"]],on="page_id",suffixes=("_a","_b"))
    if len(m)<2: return None
    delta=(m.page_f1_b-m.page_f1_a).to_numpy()
    boots=np.empty(n)
    for i in range(n): boots[i]=delta[rng.integers(0,len(delta),len(delta))].mean()
    return {"dataset":dataset,"variant_a":variant_a,"variant_b":variant_b,"shared_pages":len(m),"mean_page_F1_delta":delta.mean(),"median_page_F1_delta":np.median(delta),"improved_page_fraction":float((delta>0).mean()),"worsened_page_fraction":float((delta<0).mean()),"ci95_low":float(np.quantile(boots,.025)),"ci95_high":float(np.quantile(boots,.975))}

paired=[]
for a,b in [("original_fast","hybrid_original"),("original_fast","llm_full_html"),("hybrid_original","llm_full_html")]:
    out=paired_page_delta(a,b)
    if out: paired.append(out)
paired_effects=pd.DataFrame(paired)
paired_effects.to_csv(TABLE_DIR / "paired_page_effects_industrial.csv",index=False)
display(paired_effects)

,dataset,variant,development_group,bootstrap_mean_F1,ci95_low,ci95_high,n_pages
0,industrial,hybrid_original,all,0.940702,0.890907,0.975765,240
1,industrial,llm_full_html,all,0.965913,0.949201,0.979990,240
2,industrial,original_fast,all,0.936014,0.885920,0.973572,240
3,robustness,hybrid_original,all,0.845935,0.814752,0.875070,200
4,robustness,llm_full_html,all,1.000000,1.000000,1.000000,200
5,robustness,original_fast,all,0.725464,0.667277,0.779905,200
6,synthetic,hybrid_original,all,0.857337,0.757396,0.940001,20
7,synthetic,llm_full_html,all,1.000000,1.000000,1.000000,20
8,synthetic,original_fast,all,0.744501,0.545407,0.896959,20
9,industrial,hybrid_original,tailored,0.948071,0.897604,0.984138,229


saved 19_industrial_f1_bootstrap_ci.png


,dataset,variant_a,variant_b,shared_pages,mean_page_F1_delta,median_page_F1_delta,improved_page_fraction,worsened_page_fraction,ci95_low,ci95_high
0,industrial,original_fast,hybrid_original,240,0.007992,0.0,0.050000,0.004167,0.001611,0.016254
1,industrial,original_fast,llm_full_html,240,0.008213,0.0,0.262500,0.112500,-0.024892,0.041115
2,industrial,hybrid_original,llm_full_html,240,0.000221,0.0,0.254167,0.112500,-0.030964,0.029413


## 13. Recovery analysis: what the hybrid fixes relative to deterministic extraction

This analysis isolates pages shared by the deterministic and hybrid arms and quantifies FN recovery, new FP cost, and the relationship between deterministic failure severity and hybrid improvement.

In [32]:
def page_summary_for(v,dataset="industrial"):
    d=pages[(pages["variant"]==v)&(pages["dataset"].astype(str)==dataset)].copy()
    cols=["page_id","entity_TP","entity_FP","entity_FN"]
    extra=[c for c in ["source","development_group","candidate_count","gt_positive_count"] if c in d.columns]
    return d[cols+extra].groupby("page_id",as_index=False).agg({**{c:"sum" for c in ["entity_TP","entity_FP","entity_FN"]},**{c:"first" for c in extra}})

det=page_summary_for("original_fast")
hyb=page_summary_for("hybrid_original")
if not det.empty and not hyb.empty:
    merged=det.merge(hyb,on="page_id",suffixes=("_det","_hyb"))
    merged["FN_recovered"]=merged["entity_FN_det"]-merged["entity_FN_hyb"]
    merged["FP_added"]=merged["entity_FP_hyb"]-merged["entity_FP_det"]
    merged["TP_gain"]=merged["entity_TP_hyb"]-merged["entity_TP_det"]
    merged.to_csv(TABLE_DIR / "hybrid_recovery_by_page.csv",index=False)
    rec=pd.DataFrame([{
        "shared_pages":len(merged),
        "deterministic_FN":merged.entity_FN_det.sum(),
        "hybrid_FN":merged.entity_FN_hyb.sum(),
        "FN_recovered":merged.FN_recovered.sum(),
        "deterministic_FP":merged.entity_FP_det.sum(),
        "hybrid_FP":merged.entity_FP_hyb.sum(),
        "net_FP_added":merged.FP_added.sum(),
        "pages_with_FN_recovery":int((merged.FN_recovered>0).sum()),
        "pages_worsened_FN":int((merged.FN_recovered<0).sum()),
    }])
    rec.to_csv(TABLE_DIR / "hybrid_recovery_summary.csv",index=False)
    display(rec)

    fig,ax=plt.subplots(figsize=(8,5))
    ax.scatter(merged["entity_FN_det"],merged["FN_recovered"],alpha=.45,s=20)
    ax.axhline(0,linewidth=1); ax.set_xlabel("Deterministic false negatives on page"); ax.set_ylabel("FN recovered by hybrid")
    ax.set_title("Hybrid recovery vs deterministic failure severity")
    savefig(fig,"20_hybrid_fn_recovery_by_page")

,shared_pages,deterministic_FN,hybrid_FN,FN_recovered,deterministic_FP,hybrid_FP,net_FP_added,pages_with_FN_recovery,pages_worsened_FN
0,240,518,466,52,273,275,2,12,0


saved 20_hybrid_fn_recovery_by_page.png


## 14. Publication summary tables

The final cell writes compact tables that can be copied into the manuscript. It also produces a machine-readable manifest of every generated artifact.

In [33]:
# Compact main table.
main_cols=[c for c in ["dataset","variant","entity_P","entity_R","entity_F1","name_F1","price_F1"] if c in overall.columns]
main=overall[main_cols].copy()
main["variant"] = main["variant"].map(DISPLAY_NAMES).fillna(main["variant"])
main.to_csv(TABLE_DIR / "paper_main_quality_table.csv",index=False)

if not dev_entity.empty:
    paper_dev=dev_entity[["variant","development_group","entity_TP","entity_FP","entity_FN","entity_P","entity_R","entity_F1"]].copy()
    paper_dev["variant"] = paper_dev["variant"].map(DISPLAY_NAMES).fillna(paper_dev["variant"])
    paper_dev.to_csv(TABLE_DIR / "paper_domain_shift_table.csv",index=False)

# Inventory all exported artifacts.
manifest=[]
for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        manifest.append({"relative_path":str(p.relative_to(OUTPUT_ROOT)),"bytes":p.stat().st_size})
pd.DataFrame(manifest).to_csv(OUTPUT_ROOT / "artifact_manifest.csv",index=False)

print(f"Generated {len(list(FIGURE_DIR.glob('*.png')))} PNG figures and {len(list(TABLE_DIR.glob('*.csv')))} CSV tables (PDF export disabled).")
print("Figures:", FIGURE_DIR)
print("Tables :", TABLE_DIR)
display(main)


Generated 25 PNG figures and 21 CSV tables.
Figures: D:\Pet_Codes\science\article-price-tracker\analytics_output\figures
Tables : D:\Pet_Codes\science\article-price-tracker\analytics_output\tables


,dataset,variant,entity_P,entity_R,entity_F1,name_F1,price_F1
0,industrial,Hybrid,0.955109,0.926231,0.940448,0.935626,0.897338
1,industrial,Full-HTML LLM,0.987920,0.945069,0.966019,0.914239,0.944675
2,industrial,Deterministic,0.955040,0.917999,0.936153,0.934539,0.892959
3,robustness,Hybrid,1.000000,0.733810,0.846471,0.846471,0.846471
4,robustness,Full-HTML LLM,1.000000,1.000000,1.000000,1.000000,1.000000
5,robustness,Deterministic,1.000000,0.570000,0.726115,0.726115,0.726115
6,synthetic,Hybrid,1.000000,0.752381,0.858696,0.858696,0.858696
7,synthetic,Full-HTML LLM,1.000000,1.000000,1.000000,1.000000,1.000000
8,synthetic,Deterministic,1.000000,0.600000,0.750000,0.750000,0.750000
